# Data Preparation

Prepare the encounter data for analysis. This section applies basic cleaning and audits the resulting CSV.

## 1. Setup and Raw Data Loading

Load the verified source encounters from the local raw CSV. The loader validates the input file, schema, and original labels.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = next(
    directory
    for directory in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preparation.data_cleaning import clean_encounters, save_cleaning_result
from src.preparation.data_loading import TARGET_COLUMN, TARGET_LABELS, load_ids_mapping, load_raw_encounters

encounters = load_raw_encounters()

## 2. Basic Data Cleaning

Apply the configured marker replacements. Each configured marker becomes an empty CSV field. The laboratory category `None` retains its literal representation.

In [ ]:
cleaned = clean_encounters(encounters)
audit = cleaned.audit
print(f"Encounters: {len(encounters.frame):,} before, {len(cleaned.frame):,} after")
print(f"Columns: {len(encounters.frame.columns):,} before, {len(cleaned.frame.columns):,} after")
print(f"Changed cells: {audit['changed_cells'].sum():,}")

rules = audit["source_marker"].ne("") | audit["preserved_literal"].ne("")
display_columns = {
    "column": "Column",
    "source_marker": "Source marker",
    "marker_before": "Markers before",
    "empty_before": "Empty fields before",
    "empty_after": "Empty fields after",
    "preserved_literal": "Literal retained",
    "literal_after": "Literal count after",
}
display(audit.loc[rules, list(display_columns)].rename(columns=display_columns))

In [ ]:
cleaned_path, audit_path = save_cleaning_result(cleaned)
print(cleaned_path.relative_to(project_root))
print(audit_path.relative_to(project_root))

## 3. Cohort Selection

Exclude the configured death and hospice discharge codes before selecting one eligible encounter per patient. The smallest numeric encounter ID identifies the representative record. The source lacks encounter timestamps for ordering records by date.

In [ ]:
from src.preparation.dataset_preparation import save_cohort_result, select_cohort

cohort = select_cohort(cleaned.frame, load_ids_mapping())
display(pd.DataFrame(cohort.flow).rename(columns={
    "stage": "Stage", "encounters": "Encounters", "patients": "Patients",
    "removed": "Removed at this step",
}))

for heading, counts in (
    ("Excluded encounters by discharge code", cohort.excluded_by_code),
    ("Cohort encounters with unclear discharge descriptions", cohort.unclear_retained_by_code),
):
    print(heading)
    for code, count in counts.items():
        print(f"  {code}: {count:,}")

In [ ]:
display_columns = {
    "column": "Column",
    "cleaned_empty": "Cleaned empty fields",
    "eligible_empty": "Eligible empty fields",
    "cohort_empty": "Cohort empty fields",
}
missing = cohort.audit.loc[cohort.audit["cleaned_empty"].gt(0), list(display_columns)]
display(missing.rename(columns=display_columns))

cohort_path, cohort_audit_path = save_cohort_result(cohort)
print(cohort_path.relative_to(project_root))
print(cohort_audit_path.relative_to(project_root))

## 4. Diagnosis Grouping

Apply the fixed ICD-9 research grouping to the three diagnosis fields. The grouped fields take the place of the source diagnosis fields in the transformed table. Empty source values map to `Missing`, and values that fail parsing map to `Unmapped`.

In [ ]:
from src.preparation.dataset_preparation import group_diagnoses

grouped_cohort = group_diagnoses(cohort.frame)
print(f"Encounters: {len(cohort.frame):,} before, {len(grouped_cohort.frame):,} after")
print(f"Columns: {len(cohort.frame.columns):,} before, {len(grouped_cohort.frame.columns):,} after")
display(grouped_cohort.group_counts)

## 5. Feature Selection

Apply the approved predictor list in its configured order. Keep the encounter and patient identifiers for integrity checks and preserve the original readmission target for stratification and analysis.

In [ ]:
from src.preparation.dataset_preparation import save_feature_audit, select_predictors

selected_features = select_predictors(grouped_cohort)
feature_audit = selected_features.audit
actions = feature_audit["action"].value_counts()
print(f"Encounters: {len(selected_features.frame):,}")
print(f"Columns: {len(grouped_cohort.frame.columns):,} before, {len(selected_features.frame.columns):,} after")
for action, label in (
    ("retain_predictor", "Predictors"), ("retain_identifier", "Identifiers"),
    ("retain_target", "Original target"), ("drop_source", "Source columns dropped"),
    ("replaced_diagnosis", "Source diagnoses replaced"), ("not_created", "Columns not created"),
):
    print(f"{label}: {actions.get(action, 0)}")

retained_missing = feature_audit.loc[feature_audit["empty_after"].gt(0), ["column", "empty_after"]]
print(f"Empty fields in retained columns: {retained_missing['empty_after'].sum():,}")
display(retained_missing.rename(columns={"column": "Column", "empty_after": "Empty fields"}))

feature_audit_path = save_feature_audit(selected_features)
print(feature_audit_path.relative_to(project_root))

## 6. Target Preservation

The full prepared dataset keeps the original `readmitted` target after the 32 ordered predictors.

In [ ]:
label_counts = pd.DataFrame({
    "Cohort": cohort.frame[TARGET_COLUMN].value_counts(),
    "Prepared": selected_features.frame[TARGET_COLUMN].value_counts(),
})
display(label_counts.reindex(TARGET_LABELS, fill_value=0).rename_axis("Original label"))

## 7. Full Prepared Dataset Validation and Export

Validate the ordered columns, unique identifiers, original labels, and prepared values against the cohort. Write and read back the CSV, then record the schema, counts, and checksums in the preparation manifest.

In [ ]:
from src.preparation.dataset_preparation import save_full_prepared

full_path, manifest_path, manifest = save_full_prepared(selected_features, cohort)
summary = manifest["statistics"]
print(f"Verified dataset: {summary['rows']:,} rows and {summary['columns']} columns")
print(f"Rows with empty fields: {summary['rows_with_empty_fields']:,}")
print(full_path.relative_to(project_root))
print(manifest_path.relative_to(project_root))
print(f"CSV SHA-256: {manifest['artifact_sha256'][manifest['dataset_path']]}")